# Taller 1 - Red de Monitoreo de Calidad del Aire de Bogota.
http://rmcab.ambientebogota.gov.co/Report/stationreport

## Integrantes:
### Nombre: Christian Camilo Rosero Rodriguez, cod: 2503026-7729 
### Nombre: Carlos David Rojas Lozano, cod: 2503497-7729

## Decripción del conjunto de datos

# 1. Instalar Librerias

In [10]:
!pip install --upgrade pip # se actualiza pip, en caso de que no lo tenga actualizado
!pip install psycopg2-binary # Psycopg es un adaptador de base de datos PostgreSQL
!pip install pandas 
!pip install openpyxl


## 1.1 Importar librerias

In [2]:
import pandas as pd
from os import walk
import os, re   #expresiones regulares

# 2. Procesar conjunto de datos (Preprocesssing-dataset)

## 2.1 Unir todos los conjuntos de datos de extensión xlsx en un arreglo 


In [102]:
# unir todos los conjuntos de datos de extensión xlsx en un arreglo 
arregloDeDataSets = []
for (dirpath, dirnames, filenames) in walk('data/raw/'):
    arregloDeDataSets.extend(filenames)
    break
arregloDeDataSets

['.~lock.ca_7MA_StationsReport_20223129920.xlsx#',
 'ca_7MA_StationsReport_20223129920.xlsx',
 'ca_BOL_StationsReport_20223129119.xlsx',
 'ca_CBV_StationsReport_20223129333.xlsx',
 'ca_CDAR_StationsReport_20223129247.xlsx',
 'ca_COL_StationsReport_20223129424.xlsx',
 'ca_CSE_StationsReport_20223129152.xlsx',
 'ca_FTB_StationsReport_20223129456.xlsx',
 'ca_GYR_StationsReport_20223129531.xlsx',
 'ca_JAZ_StationsReport_20223129614.xlsx',
 'ca_KEN_StationsReport_2022312979.xlsx',
 'ca_LFR_StationsReport_20223129750.xlsx',
 'ca_MAM_StationsReport_20223129831.xlsx',
 'ca_MOV2_StationsReport_202231291026.xlsx',
 'ca_PTE_StationsReport_20223129110.xlsx',
 'ca_SCR_StationsReport_202231291142.xlsx',
 'ca_SUB_StationsReport_202231291211.xlsx',
 'ca_TUN_StationsReport_202231291241.xlsx',
 'ca_USM_StationsReport_202231291350.xlsx',
 'ca_USQ_StationsReport_202231291319.xlsx']

## 2.2  Concatenar los datsets en uno solo 


In [123]:
path = 'data/raw'
files = os.listdir(path)

df_list = []

for file in files:
    if file.endswith('.xlsx'):
        dft = pd.read_excel(os.path.join(path, file))
        dft['Station'] = re.search(r'_(.*?)_', str(file)).group(1)
        df_list.append(dft[1:])  # Agrega cada DataFrame a la lista

# Une todos los DataFrames en uno solo
df = pd.concat(df_list, ignore_index=True)

df.head()


,DateTime,PM10,CO,NO,NO2,NOX,Vel Viento,Dir Viento,Temperatura,Presion Baro,...,CO2,SO2 Envea,Vel Viento 10M,Dir Viento 10M,Temperatura 8M,Temperatura 20M,PM2.5 Flow,PM10 Flow,HR.1,Canal no activo
0,01-01-2021 01:00,52.6,----,53.384,3.428,56.817,0.4,360,14,562,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,01-01-2021 02:00,77.9,----,49.105,3.42,52.525,0,171,13.9,562,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,01-01-2021 03:00,58.5,----,47.284,4.062,51.346,0.1,170,13.8,561,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,01-01-2021 04:00,57.2,----,46.059,3.606,49.664,0.1,235,13.3,561,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,01-01-2021 05:00,53.5,----,39.851,1.625,41.475,0.1,312,13.3,562,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [124]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166440 entries, 0 to 166439
Data columns (total 27 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   DateTime         166440 non-null  object
 1   PM10             166440 non-null  object
 2   CO               166440 non-null  object
 3   NO               157680 non-null  object
 4   NO2              157680 non-null  object
 5   NOX              157680 non-null  object
 6   Vel Viento       140160 non-null  object
 7   Dir Viento       140160 non-null  object
 8   Temperatura      148920 non-null  object
 9   Presion Baro     96360 non-null   object
 10  Rad Solar        105120 non-null  object
 11  Precipitacion    105120 non-null  object
 12  HR               113880 non-null  object
 13  PM2.5            166440 non-null  object
 14  Station          166440 non-null  object
 15  SO2              122640 non-null  object
 16  OZONO            157680 non-null  object
 17  CO2       

## 2.3 Comprobar los valores nulos por columnas

In [125]:
#Contar cuantos valores nulos hay por cada columna
df.isnull().sum()

DateTime                0
PM10                    0
CO                      0
NO                   8760
NO2                  8760
NOX                  8760
Vel Viento          26280
Dir Viento          26280
Temperatura         17520
Presion Baro        70080
Rad Solar           61320
Precipitacion       61320
HR                  52560
PM2.5                   0
Station                 0
SO2                 43800
OZONO                8760
CO2                148920
SO2 Envea          157680
Vel Viento 10M     157680
Dir Viento 10M     157680
Temperatura 8M     157680
Temperatura 20M    157680
PM2.5 Flow         157680
PM10 Flow          157680
HR.1               157680
Canal no activo    157680
dtype: int64

## 2.4 Organizar y guardar en un nuevo df las columnas significativas

In [126]:
#Organizar y guardar en un nuevo df las columnas significativas
df = df[['PM10','PM2.5','NO','NO2','NOX','CO','OZONO','Station', 'DateTime']]
df.head()

,PM10,PM2.5,NO,NO2,NOX,CO,OZONO,Station,DateTime
0,52.6,37,53.384,3.428,56.817,----,NaN,7MA,01-01-2021 01:00
1,77.9,61,49.105,3.42,52.525,----,NaN,7MA,01-01-2021 02:00
2,58.5,48,47.284,4.062,51.346,----,NaN,7MA,01-01-2021 03:00
3,57.2,44,46.059,3.606,49.664,----,NaN,7MA,01-01-2021 04:00
4,53.5,39,39.851,1.625,41.475,----,NaN,7MA,01-01-2021 05:00


## 2.5 Convertir valores a numericos y sino poner NaN

In [127]:
dfCopy=df.copy()

In [128]:
cols = ['PM10', 'PM2.5', 'NO', 'NO2', 'NOX', 'CO', 'OZONO']


for col in cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166440 entries, 0 to 166439
Data columns (total 9 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   PM10      146426 non-null  float64
 1   PM2.5     151128 non-null  float64
 2   NO        138776 non-null  float64
 3   NO2       138778 non-null  float64
 4   NOX       138772 non-null  float64
 5   CO        135202 non-null  float64
 6   OZONO     134308 non-null  float64
 7   Station   166440 non-null  object 
 8   DateTime  166440 non-null  object 
dtypes: float64(7), object(2)
memory usage: 11.4+ MB


In [129]:
df.isna().sum()

PM10        20014
PM2.5       15312
NO          27664
NO2         27662
NOX         27668
CO          31238
OZONO       32132
Station         0
DateTime        0
dtype: int64

In [130]:
dfCopy.isna().sum()

PM10           0
PM2.5          0
NO          8760
NO2         8760
NOX         8760
CO             0
OZONO       8760
Station        0
DateTime       0
dtype: int64

## 2.6 Ver el tamaño del dataset 

In [131]:
len(df)

166440

## 2.7 Crear una nueva columna con filtro (False o True)

In [132]:
# True is greater than 12, bad (Pure, Not Pure)
df['Status'] = df['PM2.5']>12
statusCount=df.Status.value_counts()
df.head()


,PM10,PM2.5,NO,NO2,NOX,CO,OZONO,Station,DateTime,Status
0,52.6,37.0,53.384,3.428,56.817,NaN,NaN,7MA,01-01-2021 01:00,True
1,77.9,61.0,49.105,3.420,52.525,NaN,NaN,7MA,01-01-2021 02:00,True
2,58.5,48.0,47.284,4.062,51.346,NaN,NaN,7MA,01-01-2021 03:00,True
3,57.2,44.0,46.059,3.606,49.664,NaN,NaN,7MA,01-01-2021 04:00,True
4,53.5,39.0,39.851,1.625,41.475,NaN,NaN,7MA,01-01-2021 05:00,True


In [133]:
print(statusCount)

Status
False    91098
True     75342
Name: count, dtype: int64


## 2.8 Guardar todo el proceso en un nuevo dataset

In [134]:
# we need another column to specify stations
csv_file= "data/clean_data_final.csv"
df.to_csv(csv_file, index = False)
print(f"\nSe procesaron {len(df)} registros y se guardaron en {csv_file}'")



Se procesaron 166440 registros y se guardaron en data/clean_data_final.csv'


# 3. Integrar nuevo dataset (Stations)

## 3.1 Cargar dataset de stations

In [135]:
#cargar el df dataset_with_missing.csv   data/dataset_with_missing.csv'
#cargar el df stations_loc.csv data/stations_loc.csv')

#Aquí su código 

dataWithMissing = pd.read_csv("data/dataset_with_missing.csv")
dataStationsLoc= pd.read_csv("data/stations_loc.csv")

### 3.1.1 Seleccionar algunas columnas del dataset stations

In [136]:

stations = dataStationsLoc[['Sigla', 'Latitud', 'Longitud','Localidad','estacion']] 


### 3.1.2 Cambiar el nombre de una columna 

In [137]:

stations = stations.rename(columns={'Sigla': 'Station'}) #Columna Sigla por Station
stations = stations.rename(columns={'estacion': 'Nombre'}) 
stations.head()

,Station,Latitud,Longitud,Localidad,Nombre
0,GYR,"4°47'01.5""N","74°02'38.9""W",Suba,guaymaral
1,USQ,"4°42'37.26""N","74°1'49.50""W",Usaquén,usaquen
2,SUB,"4°45'40.49""N","74° 5'36.46""W",Suba,suba
3,BOL,"4°44'08.9""N","74°07'33.2""W",Engativá,bolivia
4,LFR,"4°41'26.52""N","74°4'56.94""W",Engativá,las_ferias


## 3.2 Convertir las coordenadas de texto a decimales

In [138]:
import re

def dms2dd(degrees, minutes, seconds, direction):
    dd = float(degrees) + float(minutes)/60 + float(seconds)/(60*60);
    if direction == 'S' or direction == 'W':
        dd *= -1
    return dd;

def dd2dms(deg):
    d = int(deg)
    md = abs(deg - d) * 60
    m = int(md)
    sd = (md - m) * 60
    return [d, m, sd]

def parse_dms(coor):
    parts = re.split('[^\d\w]+', coor)
    dec_coor = dms2dd(parts[0], parts[1], float(parts[2]+'.'+parts[2]), parts[4])
    return dec_coor



In [139]:
#Aplicar la funcion parse_dms a la columna Latitud y Longitud
stations['Latitud'] = stations['Latitud'].apply(parse_dms)
stations['Longitud'] = stations['Longitud'].apply(parse_dms)


In [140]:
print(stations)

   Station   Latitud   Longitud       Localidad                   Nombre
0      GYR  4.783614 -74.043994            Suba                guaymaral
1      USQ  4.710381 -74.030414         Usaquén                  usaquen
2      SUB  4.761222 -74.093433            Suba                     suba
3      BOL  4.735578 -74.125925        Engativá                  bolivia
4      LFR  4.690628 -74.082378        Engativá               las_ferias
5     CDAR  4.658417 -74.083944        Engativá  centro_alto_rendimiento
6      7MA  4.645117 -74.061503       Chapinero                movil_7ma
7      MAM  4.625364 -74.066972        Santa Fe              minambiente
8      FTB  4.678169 -74.143714        Fontibón                 fontibon
9      PTE  4.631817 -74.117278   Puente Aranda            pueste_aranda
10     KEN  4.625083 -74.161222         Kennedy                  kennedy
11     CSE  4.595958 -74.148483         Kennedy       carvajal_sevillana
12     TUN  4.576206 -74.130975      Tunjuelito    

## 3.3 Integrar el dataset df con el dataset stations

In [141]:
df = pd.merge(df, stations, on='Station', how='inner')
df.head(3)

,PM10,PM2.5,NO,NO2,NOX,CO,OZONO,Station,DateTime,Status,Latitud,Longitud,Localidad,Nombre
0,52.6,37.0,53.384,3.428,56.817,NaN,NaN,7MA,01-01-2021 01:00,True,4.645117,-74.061503,Chapinero,movil_7ma
1,77.9,61.0,49.105,3.420,52.525,NaN,NaN,7MA,01-01-2021 02:00,True,4.645117,-74.061503,Chapinero,movil_7ma
2,58.5,48.0,47.284,4.062,51.346,NaN,NaN,7MA,01-01-2021 03:00,True,4.645117,-74.061503,Chapinero,movil_7ma


## 3.4 Convertir hora (DateTime)

In [142]:
#Puede ver que en la columna 'DateTime', 
#la información sobre la fecha y la hora se dan juntas. 
#Por lo tanto, extraerá la información de tiempo.

def replace24(datetimex):
    return datetimex.replace('24:00', '00:00') #cambia de horarios

In [143]:
# Esta celda extraerá información de la columna 'datetime' y 
#generará columnas de meses, días o semanas y horas
df['DateTime'] = df['DateTime'].apply(replace24)
df['DateTime'] = pd.to_datetime(df['DateTime'], dayfirst=True)
df['month'] = pd.DatetimeIndex(df['DateTime']).month
df['day_week'] = pd.DatetimeIndex(df['DateTime']).weekday
df['day_month'] = pd.DatetimeIndex(df['DateTime']).day
df['hour'] = pd.DatetimeIndex(df['DateTime']).hour
df.loc[df['hour']==0,'hour'] = 24

In [144]:
df

,PM10,PM2.5,NO,NO2,NOX,CO,OZONO,Station,DateTime,Status,Latitud,Longitud,Localidad,Nombre,month,day_week,day_month,hour
0,52.6,37.0,53.384,3.428,56.817,NaN,NaN,7MA,2021-01-01 01:00:00,True,4.645117,-74.061503,Chapinero,movil_7ma,1,4,1,1
1,77.9,61.0,49.105,3.420,52.525,NaN,NaN,7MA,2021-01-01 02:00:00,True,4.645117,-74.061503,Chapinero,movil_7ma,1,4,1,2
2,58.5,48.0,47.284,4.062,51.346,NaN,NaN,7MA,2021-01-01 03:00:00,True,4.645117,-74.061503,Chapinero,movil_7ma,1,4,1,3
3,57.2,44.0,46.059,3.606,49.664,NaN,NaN,7MA,2021-01-01 04:00:00,True,4.645117,-74.061503,Chapinero,movil_7ma,1,4,1,4
4,53.5,39.0,39.851,1.625,41.475,NaN,NaN,7MA,2021-01-01 05:00:00,True,4.645117,-74.061503,Chapinero,movil_7ma,1,4,1,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166435,25.3,18.0,1.404,3.688,5.091,0.26467,29.773,USQ,2021-12-31 20:00:00,True,4.710381,-74.030414,Usaquén,usaquen,12,4,31,20
166436,27.7,15.0,0.244,3.205,3.449,0.27138,28.059,USQ,2021-12-31 21:00:00,True,4.710381,-74.030414,Usaquén,usaquen,12,4,31,21
166437,25.3,20.0,0.880,4.648,5.529,0.30159,25.561,USQ,2021-12-31 22:00:00,True,4.710381,-74.030414,Usaquén,usaquen,12,4,31,22
166438,24.0,17.0,1.697,8.596,10.293,0.34977,20.539,USQ,2021-12-31 23:00:00,True,4.710381,-74.030414,Usaquén,usaquen,12,4,31,23


## 3.5 Guardar todo el proceso en un nuevo dataset

In [145]:
df.to_csv("data/dataset_with_geo_missing.csv", index = False)

# 4. Limpieza de datos

In [146]:
# Varias columnas tienen valores perdidos en el dataset

df.isnull().sum()

PM10         20014
PM2.5        15312
NO           27664
NO2          27662
NOX          27668
CO           31238
OZONO        32132
Station          0
DateTime         0
Status           0
Latitud          0
Longitud         0
Localidad        0
Nombre           0
month            0
day_week         0
day_month        0
hour             0
dtype: int64

## 4.1 Remplazar los valores NaN en cada columna con su media

In [147]:
df['PM10'].fillna((df['PM10'].mean()), inplace=True)
df['PM2.5'].fillna((df['PM2.5'].mean()), inplace=True)
df['NO'].fillna((df['NO'].mean()), inplace=True)
df['CO'].fillna((df['CO'].mean()), inplace=True)
df['NO2'].fillna((df['NO2'].mean()), inplace=True)
df['NOX'].fillna((df['NOX'].mean()), inplace=True)
df['OZONO'].fillna((df['OZONO'].mean()), inplace=True)

#hagalo para PM2.5, CO, NO, NO2, NOx, OZONO
#Aquí su código 

In [148]:
# compruebe --- ya no hay valores perdidos en el dataset

df.isnull().sum()

PM10         0
PM2.5        0
NO           0
NO2          0
NOX          0
CO           0
OZONO        0
Station      0
DateTime     0
Status       0
Latitud      0
Longitud     0
Localidad    0
Nombre       0
month        0
day_week     0
day_month    0
hour         0
dtype: int64

## 4.2 Guardar todo el proceso en un nuevo dataset

In [149]:
pathDataSetMean="data/dataset_final_clean_mean.csv"
df.to_csv(pathDataSetMean, index = False)
dfCopyMean=df.copy()

In [150]:
print(f"\nSe procesaron {len(dfCopyMean)} registros y se guardaron en {pathDataSetMean}'")


Se procesaron 166440 registros y se guardaron en data/dataset_final_clean_mean.csv'


# 5. Creando el esquema de la bodega de datos 

In [151]:
df = pd.read_csv('data/dataset_final_clean_mean.csv')
df.head(10)

,PM10,PM2.5,NO,NO2,NOX,CO,OZONO,Station,DateTime,Status,Latitud,Longitud,Localidad,Nombre,month,day_week,day_month,hour
0,52.6,37.0,53.384,3.428,56.817,0.691233,11.782188,7MA,2021-01-01 01:00:00,True,4.645117,-74.061503,Chapinero,movil_7ma,1,4,1,1
1,77.9,61.0,49.105,3.420,52.525,0.691233,11.782188,7MA,2021-01-01 02:00:00,True,4.645117,-74.061503,Chapinero,movil_7ma,1,4,1,2
2,58.5,48.0,47.284,4.062,51.346,0.691233,11.782188,7MA,2021-01-01 03:00:00,True,4.645117,-74.061503,Chapinero,movil_7ma,1,4,1,3
3,57.2,44.0,46.059,3.606,49.664,0.691233,11.782188,7MA,2021-01-01 04:00:00,True,4.645117,-74.061503,Chapinero,movil_7ma,1,4,1,4
4,53.5,39.0,39.851,1.625,41.475,0.691233,11.782188,7MA,2021-01-01 05:00:00,True,4.645117,-74.061503,Chapinero,movil_7ma,1,4,1,5
5,22.3,15.0,39.956,0.677,40.633,0.691233,11.782188,7MA,2021-01-01 06:00:00,True,4.645117,-74.061503,Chapinero,movil_7ma,1,4,1,6
6,18.3,8.0,40.870,1.069,41.940,0.691233,11.782188,7MA,2021-01-01 07:00:00,False,4.645117,-74.061503,Chapinero,movil_7ma,1,4,1,7
7,23.2,10.0,42.630,1.615,44.245,0.691233,11.782188,7MA,2021-01-01 08:00:00,False,4.645117,-74.061503,Chapinero,movil_7ma,1,4,1,8
8,24.4,12.0,38.317,12.145,50.463,0.691233,11.782188,7MA,2021-01-01 09:00:00,False,4.645117,-74.061503,Chapinero,movil_7ma,1,4,1,9
9,15.2,9.0,37.499,10.979,48.478,0.691233,11.782188,7MA,2021-01-01 10:00:00,False,4.645117,-74.061503,Chapinero,movil_7ma,1,4,1,10


## 5.1 Se crea un nuevo DataFrame con los datos de los polutantes

In [152]:
df_polutante=pd.DataFrame(df, columns=["PM10", "PM2.5", "NO", "NO2", "NOX", "CO", "OZONO"]) # se crea un nuevo DataFrame unicamente con los polutantes
df['id_polutante'] = df.index+1 # se crea una nueva columna que identificará a cada polutante
df_polutante

,PM10,PM2.5,NO,NO2,NOX,CO,OZONO
0,52.6,37.0,53.384,3.428,56.817,0.691233,11.782188
1,77.9,61.0,49.105,3.420,52.525,0.691233,11.782188
2,58.5,48.0,47.284,4.062,51.346,0.691233,11.782188
3,57.2,44.0,46.059,3.606,49.664,0.691233,11.782188
4,53.5,39.0,39.851,1.625,41.475,0.691233,11.782188
...,...,...,...,...,...,...,...
166435,25.3,18.0,1.404,3.688,5.091,0.264670,29.773000
166436,27.7,15.0,0.244,3.205,3.449,0.271380,28.059000
166437,25.3,20.0,0.880,4.648,5.529,0.301590,25.561000
166438,24.0,17.0,1.697,8.596,10.293,0.349770,20.539000


## 5.2 Se crea un nuevo dataset con los datos de las estaciones y se eliminan las filas repetidas

In [154]:
df_estacion=pd.DataFrame(df, columns=["Nombre", "Station", "Localidad", "Latitud", "Longitud"]) # se crea un nuevo DataFrame unicamente con los datos de las estaciones
df_estacion = df_estacion.drop_duplicates() # se eliminan las filas repetidas
df_estacion = df_estacion.rename(columns={'Station': 'Sigla'}) # se cambia el nombre a la columna Station por Sigla
df_estacion

,Nombre,Sigla,Localidad,Latitud,Longitud
0,movil_7ma,7MA,Chapinero,4.645117,-74.061503
8760,bolivia,BOL,Engativá,4.735578,-74.125925
17520,ciudad_bolivar,CBV,Ciudad Bolívar,4.577889,-74.166272
26280,centro_alto_rendimiento,CDAR,Engativá,4.658417,-74.083944
35040,colina,COL,Suba,4.736981,-74.069472
43800,carvajal_sevillana,CSE,Kennedy,4.595958,-74.148483
52560,fontibon,FTB,Fontibón,4.678169,-74.143714
61320,guaymaral,GYR,Suba,4.783614,-74.043994
70080,el_jazmin,JAZ,Puente Aranda,4.608417,-74.114869
78840,kennedy,KEN,Kennedy,4.625083,-74.161222


In [155]:
# se remplaza la columna sigla, dejando unicamente los datos unicos y luego se le cambia el nombre
# a la misma por id_estacion
count = 1
for index, row in df_estacion.iterrows():
    df = df.replace({row["Sigla"]: count})
    count += 1
df = df.rename(columns={'Station': 'id_estacion'})
df

,PM10,PM2.5,NO,NO2,NOX,CO,OZONO,id_estacion,DateTime,Status,Latitud,Longitud,Localidad,Nombre,month,day_week,day_month,hour,id_polutante
0,52.6,37.0,53.384,3.428,56.817,0.691233,11.782188,1,2021-01-01 01:00:00,True,4.645117,-74.061503,Chapinero,movil_7ma,1,4,1,1,1
1,77.9,61.0,49.105,3.420,52.525,0.691233,11.782188,1,2021-01-01 02:00:00,True,4.645117,-74.061503,Chapinero,movil_7ma,1,4,1,2,2
2,58.5,48.0,47.284,4.062,51.346,0.691233,11.782188,1,2021-01-01 03:00:00,True,4.645117,-74.061503,Chapinero,movil_7ma,1,4,1,3,3
3,57.2,44.0,46.059,3.606,49.664,0.691233,11.782188,1,2021-01-01 04:00:00,True,4.645117,-74.061503,Chapinero,movil_7ma,1,4,1,4,4
4,53.5,39.0,39.851,1.625,41.475,0.691233,11.782188,1,2021-01-01 05:00:00,True,4.645117,-74.061503,Chapinero,movil_7ma,1,4,1,5,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166435,25.3,18.0,1.404,3.688,5.091,0.264670,29.773000,19,2021-12-31 20:00:00,True,4.710381,-74.030414,Usaquén,usaquen,12,4,31,20,166436
166436,27.7,15.0,0.244,3.205,3.449,0.271380,28.059000,19,2021-12-31 21:00:00,True,4.710381,-74.030414,Usaquén,usaquen,12,4,31,21,166437
166437,25.3,20.0,0.880,4.648,5.529,0.301590,25.561000,19,2021-12-31 22:00:00,True,4.710381,-74.030414,Usaquén,usaquen,12,4,31,22,166438
166438,24.0,17.0,1.697,8.596,10.293,0.349770,20.539000,19,2021-12-31 23:00:00,True,4.710381,-74.030414,Usaquén,usaquen,12,4,31,23,166439


## 5.3 Se crea un nuevo DataFrame con los datos de las fechas.

In [156]:
# se crea un nuevo DataFrame y se eliminan los valores repetidos
df_fecha=pd.DataFrame(df, columns=["DateTime"]) # se crea un nuevo DataFrame con las fechas y horas 
#df_fecha["DateTime"].value_counts() # !TODO Borrar?
df_fecha = df_fecha.drop_duplicates() # se eliminan las filas repetidas
df_fecha

,DateTime
0,2021-01-01 01:00:00
1,2021-01-01 02:00:00
2,2021-01-01 03:00:00
3,2021-01-01 04:00:00
4,2021-01-01 05:00:00
...,...
8755,2021-12-31 20:00:00
8756,2021-12-31 21:00:00
8757,2021-12-31 22:00:00
8758,2021-12-31 23:00:00


In [157]:
import re # libreria para el uso de expresiones regulares
from pandas.tseries.holiday import Holiday, AbstractHolidayCalendar
from datetime import datetime

# definimos los días festivo para el año en que se tomaron los datos
class EsBusinessCalendar(AbstractHolidayCalendar):
   rules = [ # se definen los festivos, teniendo en cuenta los horarios festivos para el año de 2021 en Colombia
     Holiday('Año Nuevo', month=1, day=1),
     Holiday('Día de los Reyes Magos', month=1, day=11),
     Holiday('Día de San José', month=3, day=22),
     Holiday('Jueves Santo', month=4, day=1),
     Holiday('Viernes Santo', month=4, day=2),
     Holiday('Día del Trabajador', month=5, day=1),
     Holiday('Día de la Ascensión', month=5, day=17),
     Holiday('Corpus Christi', month=6, day=7),
     Holiday('Sagrado Corazón', month=6, day=14),
     Holiday('San Pedro y San Pablo ', month=7, day=5),
     Holiday('Día de la Independencia', month=7, day=20),
     Holiday('Batalla de Boyacá', month=8, day=7),
     Holiday('Asunción de la Virgen', month=8, day=16),
     Holiday('Celebración del Día de la Raza', month=10, day=18),
     Holiday('Día de todos los Santos', month=11, day=1),
     Holiday('Independencia de Cartagena', month=11, day=15),
     Holiday('Inmaculada Concepción', month=12, day=8),    
     Holiday('Navidad', month=12, day=25)
   ]

calendar_festivos = EsBusinessCalendar() # se instancia el objeto
calendar_festivos = calendar_festivos.holidays(start='2021-01-01', end='2021-12-31') # se define el rango de tiempo en que se tendran en cuenta los festivos (anho 2021)

dias = []
meses = []
anhos = []
horas = []
fin_semana = []
festivo = []


for index, row in df_fecha.iterrows(): # se recorre cada fila del DataFrame
    list_fecha = re.split("[\-\s]", row['DateTime']) # 
    dias.append(list_fecha[2])
    meses.append(list_fecha[1])
    anhos.append(list_fecha[0])
    horas.append(list_fecha[3])

    hora = row['DateTime'].split(" ")
    hora[0] = datetime.strptime(hora[0], '%Y-%m-%d')
    if hora[0].weekday() < 5 : # condicción para determinar si el día está comprendido entre lues-viernes
        fin_semana.append(False)
    else:
        fin_semana.append(True)
    if hora[0] in calendar_festivos: # condicción para determinar si el día hace parte de los días festivos del año
        festivo.append(True)
    else: 
        festivo.append(False)

dict_fechas = { 'dia': dias, 'mes': meses, 'anho': anhos, 'hora': horas, 'fin_semana': fin_semana, 'festivo': festivo}
df_fechas = pd.DataFrame(data=dict_fechas)
df_fechas


,dia,mes,anho,hora,fin_semana,festivo
0,01,01,2021,01:00:00,False,True
1,01,01,2021,02:00:00,False,True
2,01,01,2021,03:00:00,False,True
3,01,01,2021,04:00:00,False,True
4,01,01,2021,05:00:00,False,True
...,...,...,...,...,...,...
8755,31,12,2021,20:00:00,False,False
8756,31,12,2021,21:00:00,False,False
8757,31,12,2021,22:00:00,False,False
8758,31,12,2021,23:00:00,False,False


## 5.4 Creación de la columna id_tiempo para identificar cada una de las fechas en el DF principal

In [161]:
count = 1
# se recorre cada fecha de df_fecha

for index, row in df_fecha.iterrows(): 
# por cada fila de df_fecha se crea un nuevo identificador en el DataFrame principal
    df.loc[df['DateTime'] == row["DateTime"], 'DateTime'] = count     
    count += 1
df = df.rename(columns={'DateTime': 'id_tiempo'})

df

,PM10,PM2.5,NO,NO2,NOX,CO,OZONO,id_estacion,id_tiempo,Status,Latitud,Longitud,Localidad,Nombre,month,day_week,day_month,hour,id_polutante
0,52.6,37.0,53.384,3.428,56.817,0.691233,11.782188,1,1,True,4.645117,-74.061503,Chapinero,movil_7ma,1,4,1,1,1
1,77.9,61.0,49.105,3.420,52.525,0.691233,11.782188,1,2,True,4.645117,-74.061503,Chapinero,movil_7ma,1,4,1,2,2
2,58.5,48.0,47.284,4.062,51.346,0.691233,11.782188,1,3,True,4.645117,-74.061503,Chapinero,movil_7ma,1,4,1,3,3
3,57.2,44.0,46.059,3.606,49.664,0.691233,11.782188,1,4,True,4.645117,-74.061503,Chapinero,movil_7ma,1,4,1,4,4
4,53.5,39.0,39.851,1.625,41.475,0.691233,11.782188,1,5,True,4.645117,-74.061503,Chapinero,movil_7ma,1,4,1,5,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166435,25.3,18.0,1.404,3.688,5.091,0.264670,29.773000,19,8756,True,4.710381,-74.030414,Usaquén,usaquen,12,4,31,20,166436
166436,27.7,15.0,0.244,3.205,3.449,0.271380,28.059000,19,8757,True,4.710381,-74.030414,Usaquén,usaquen,12,4,31,21,166437
166437,25.3,20.0,0.880,4.648,5.529,0.301590,25.561000,19,8758,True,4.710381,-74.030414,Usaquén,usaquen,12,4,31,22,166438
166438,24.0,17.0,1.697,8.596,10.293,0.349770,20.539000,19,8759,True,4.710381,-74.030414,Usaquén,usaquen,12,4,31,23,166439


In [162]:
# finalmente se crea un DataFrame, el cual se relaciona con los DataFrame creados anteriormente por medio de sus identificadores 
df_fact_medidad=pd.DataFrame(df, columns=["id_estacion", "id_tiempo", "id_polutante"])
df_fact_medidad

,id_estacion,id_tiempo,id_polutante
0,1,1,1
1,1,2,2
2,1,3,3
3,1,4,4
4,1,5,5
...,...,...,...
166435,19,8756,166436
166436,19,8757,166437
166437,19,8758,166438
166438,19,8759,166439


## 5.6Creación de las tablas en Postgres

In [163]:
# Es necesario tener instaladas las librerias 
!pip install python-dotenv
!pip install psycopg2-binary

In [164]:
from conexion import new_model

# se llama a la función new_model (archivo conexion.py) con cada uno de los DataFrame para crear las tablas correspondientes
df_fact_medidad = df_fact_medidad.astype(int)
new_model(df_estacion, "dim_estacion")

#Cree las tablas para dim_polutante, dim_polutante, fact_medidad



In [165]:
new_model(df_polutante, "dim_polutante")

In [166]:
new_model(df_fact_medidad, "fact_medidad")

In [167]:
new_model(df_fechas, "dim_tiempo")